# Практика · BERT> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)> ⏱ Зошит **передтреновує дві маленькі моделі BERT** і будує на них класифікатор.> Заміряно: **близько двох з половиною хвилин процесорного часу** (157 с) на> чотирьох ядрах без відеокарти, **в один потік**. Загалом від першої клітинки> до останньої на вільній машині виходить близько трьох хвилин. На завантаженій> стінного часу буде більше — зошит друкує і навантаження, і власний> процесорний час останньою клітинкою.## Задача зошитаУ нас є справжній український текст без жодних міток — переклади інтерфейсів,які лежать у системі. І є практична задача: **чи повідомляє цей рядок пропомилку**. Міток для неї мало, розмічати дорого.Зробимо так, як робить BERT:1. **передтренуємо** енкодер на тексті **без міток**, ховаючи в ньому слова й   змушуючи модель їх відновлювати;2. **заморозимо** його й побудуємо поверх звичайну логістичну регресію на   мітках;3. **порівняємо** з тим самим енкодером, який нічого не вчив, і з TF-IDF.Дорогою перевіримо руками дві речі, про які в лекції сказано словами:як працює WordPiece і що станеться, якщо замість `[MASK]` завжди лишати словона місці.⚠️ **Готових ваг BERT у нас немає** — мережі в зошитах курсу заборонено, а кешпорожній. Тому все, що тут є, ми навчаємо самі. Це навчальний макет на мільйоніз гаком ваг; справжній BERT більший у вісімдесят разів і бачив корпус більшийу шість тисяч разів. Чого від макета не варто чекати — сказано в кінці.

## 0 · Середовище й чому ми фіксуємо потокиЗошит міряє час, а час на спільній машині бреше двома різними способами.**Стінний годинник** показує, скільки минуло реального часу. Якщо поручрахується щось іще, він покаже більше — і число нічого не варте.**Процесорний годинник** (`time.process_time()`) рахує лише той час, колипроцесор працював над нашою програмою. Він чесніший, але має власну пастку:якщо бібліотека лінійної алгебри розкладає роботу на кілька потоків, то потоки,які **чекають**, теж рахуються як робота. Тому спершу фіксуємо один потік — іробимо це **до** імпорту `numpy`, бо змінні середовища читаються призавантаженні бібліотеки.

In [ ]:
import os
# ⚠️ ці три рядки мусять стояти ДО імпорту numpy і torch — інакше
# бібліотеки вже запустять свої потоки, і процесорний час стане брехливим
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import glob, gettext, re, math, gc, time, sys
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
ЗАПУСК = time.process_time()          # звідси рахуємо власний час зошита

print('python              ', sys.version.split()[0])
print('torch               ', torch.__version__)
print('ядер у машині       ', os.cpu_count())
print('потоків у torch     ', torch.get_num_threads())
print('відеокарта          ', 'є' if torch.cuda.is_available() else 'немає')
print('навантаження машини ', round(os.getloadavg()[0], 2))

## 1 · КорпусБеремо українські переклади інтерфейсів із `/usr/share/locale/uk/LC_MESSAGES/`.Це справжня українська мова, вона вже лежить на машині, і вона однакова відзапуску до запуску.Кожен запис каталогу — це пара «англійський оригінал → український переклад».Нам знадобляться обидва боки, але для різного:- **український** бік дає текст, на якому вчиться модель;- **англійський** бік дасть мітку «це повідомлення про помилку» — регуляркою по  восьми типових словах. Модель англійського боку не бачить ніколи, тож мітка  не є круговою.Ділити текст на слова будемо канонічним для курсу регулярним виразом, у якомуапостроф — звʼязка всередині слова.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
ERROR_WORDS = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)

def load_corpus():
    '''Повертає трійки (програма, англійський оригінал, слова перекладу).'''
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                    # зламаний каталог просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                out.append((program, source, re.findall(TOKEN_PATTERN, target.lower())))
    return out

corpus = load_corpus()
if len(corpus) < 1000:
    raise RuntimeError('українських каталогів у системі майже немає — '
                       'зошитові немає на чому працювати')

# надто короткі й надто довгі рядки прибираємо: у рядку з одного слова
# нема чого маскувати, а хвіст на півтисячі слів роздув би пачки заповнювачем
rows = [r for r in corpus if 2 <= len(r[2]) <= 30]
lengths = np.array([len(r[2]) for r in rows])

print('записів у каталогах  ', len(corpus))
print('лишилось після відсіву', len(rows))
print('слововживань         ', int(lengths.sum()))
print('різних словоформ     ', len({w for r in rows for w in r[2]}))
print('медіана довжини      ', int(np.median(lengths)), 'слів')
print('програм              ', len({r[0] for r in rows}))

**Запамʼятай медіану** — половина рядків коротша за неї. Далі це виявитьсяважливим двічі: коротка фраза дає мало замаскованих позицій, і вона ж робитьнашу пару речень короткою.## 2 · Три частини: навчальна, відкладена, перевірнаДілимо на **три** частини, а не на дві:- **навчальна** — на ній модель передтреновується й на ній же вчиться зонд;- **відкладена** — на ній добирають гіперпараметри. Ми свою швидкість навчання  вже дібрали окремим прогоном (сітка з пʼяти значень, мінімум усередині), тож  тут вона потрібна лише для того, щоб поділ був такий самий, як у заміру;- **перевірна** — на ній оголошується результат. Модель не бачить її ніколи.⚠️ Рядки лежать **у порядку програм**. Якби ми взяли «перші десять відсотків»,це була б не менша вибірка, а вужчий домен. Тому перемішуємо з фіксованимзерном.

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(rows))
n_test = len(rows) // 10
test_positions = set(order[:n_test].tolist())

train_all = [rows[i] for i in range(len(rows)) if i not in test_positions]
test_rows = [rows[i] for i in range(len(rows)) if i in test_positions]

# відкладена — 5 % навчальної частини, окреме зерно
rng_dev = np.random.default_rng(1)
order_dev = rng_dev.permutation(len(train_all))
n_dev = int(round(len(train_all) * 0.05))
dev_positions = set(order_dev[:n_dev].tolist())

dev_rows = [train_all[i] for i in range(len(train_all)) if i in dev_positions]
train_rows = [train_all[i] for i in range(len(train_all)) if i not in dev_positions]

train_labels = np.array([1 if ERROR_WORDS.search(r[1]) else 0 for r in train_rows])
test_labels = np.array([1 if ERROR_WORDS.search(r[1]) else 0 for r in test_rows])

print('навчальних  ', len(train_rows))
print('відкладених ', len(dev_rows))
print('перевірних  ', len(test_rows))
print('слововживань у навчальній частині', sum(len(r[2]) for r in train_rows))
print()
print('частка класу «помилка» у навчальній', round(float(train_labels.mean()), 4))
print('частка класу «помилка» у перевірній', round(float(test_labels.mean()), 4))

## 3 · WordPiece своїми рукамиBERT працює не зі словами, а з **субсловами**. Словник WordPiece містить шматкидвох сортів: **початкові** (з них слово може починатись) і **продовження**,позначені двома ґратками спереду — `##ування`, `##ів`.Поділ робиться **жадібним найдовшим збігом зліва**: беремо найдовший префікс,який є у словнику, відрізаємо, шукаємо далі вже серед продовжень.Спершу навчимо словник бібліотекою, а потім напишемо сам поділ самі — і звіримоз бібліотечним. Це найкорисніша перевірка в зошиті: усередині токенізатора немаємагії, там десять рядків циклу.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

SPECIAL = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]']

# навчаємо словник WordPiece на українському боці навчальної частини
piece_tokenizer = Tokenizer(models.WordPiece(unk_token='[UNK]',
                                             max_input_chars_per_word=100))
# ⚠️ саме WhitespaceSplit, а не Whitespace: другий різав би слово по апострофу,
# і «зʼєднання» перестало б бути одним словом ще до всякого WordPiece
piece_tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
trainer = trainers.WordPieceTrainer(vocab_size=4000, special_tokens=SPECIAL,
                                    min_frequency=2)
piece_tokenizer.train_from_iterator((' '.join(r[2]) for r in train_rows), trainer)

piece_vocab = piece_tokenizer.get_vocab()
print('шматків у словнику', len(piece_vocab))
print('із них продовжень ', sum(1 for p in piece_vocab if p.startswith('##')))

Тепер наш власний поділ. Алгоритм рівно такий, як описано вище: доки слово нескінчилось, шукаємо найдовший шматок, який підходить із поточного місця.

In [ ]:
def wordpiece_split(word, vocab, unk='[UNK]'):
    '''Жадібний найдовший збіг зліва. Повертає список шматків.'''
    pieces = []
    start = 0
    while start < len(word):
        end = len(word)
        found = None
        while end > start:
            piece = word[start:end]
            if start > 0:               # не початок слова -> шукаємо продовження
                piece = '##' + piece
            if piece in vocab:
                found = piece
                break
            end -= 1                    # не знайшли — вкорочуємо на літеру
        if found is None:
            return [unk]                # жоден шматок не підійшов: усе слово невідоме
        pieces.append(found)
        start = end
    return pieces

# показуємо на словах, які цікаві саме для української
for word in ['файл', 'файлів', 'налаштування', 'налаштувань', 'зберегти',
             'криптографія']:
    print('%-14s -> %s' % (word, ' | '.join(wordpiece_split(word, piece_vocab))))

А тепер перевірка: чи збігається наш поділ із бібліотечним **на всіх** словахкорпусу. Якщо збігається — значить, ми справді зрозуміли алгоритм, а не написалищось схоже.

In [ ]:
different = []
all_words = sorted({w for r in train_rows for w in r[2]})
for word in all_words:
    ours = wordpiece_split(word, piece_vocab)
    theirs = piece_tokenizer.encode(word).tokens
    if ours != theirs:
        different.append((word, ours, theirs))

print('перевірено словоформ', len(all_words))
print('розбіжностей        ', len(different))
assert not different, different[:5]
print('✅ наш поділ збігається з бібліотечним на кожному слові')

pieces_total = sum(len(wordpiece_split(w, piece_vocab)) for w in all_words)
print()
print('шматків на словоформу в середньому: %.4f' % (pieces_total / len(all_words)))
whole = sum(1 for w in all_words if len(wordpiece_split(w, piece_vocab)) == 1)
print('словоформ, що лишились цілими:      %.4f' % (whole / len(all_words)))

⚠️ **Числа двох останніх рядків від запуску до запуску трохи гуляють**, і це нездогад. Перевірмо прямо: навчимо словник **удруге** на тих самих даних із тимисамими налаштуваннями й звірмо два результати.

In [ ]:
second_tokenizer = Tokenizer(models.WordPiece(unk_token='[UNK]',
                                              max_input_chars_per_word=100))
second_tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
second_tokenizer.train_from_iterator((' '.join(r[2]) for r in train_rows),
                                     trainers.WordPieceTrainer(
                                         vocab_size=4000, special_tokens=SPECIAL,
                                         min_frequency=2))
second_vocab = second_tokenizer.get_vocab()

only_first = set(piece_vocab) - set(second_vocab)
only_second = set(second_vocab) - set(piece_vocab)
print('розмір першого словника ', len(piece_vocab))
print('розмір другого словника ', len(second_vocab))
print('шматків лише в першому  ', len(only_first))
print('шматків лише в другому  ', len(only_second))
print('словники збігаються цілком (разом із номерами):', piece_vocab == second_vocab)

# і найважливіше: чи однаково два словники ділять слова
other_split = sum(1 for w in all_words
                  if wordpiece_split(w, piece_vocab) != wordpiece_split(w, second_vocab))
print('словоформ, поділених по-різному:', other_split, 'із', len(all_words))

Подивись на передостанній рядок: два словники, навчені однаковим кодом наоднакових даних, **не збіглися**. Найчастіше набір шматків виходить той самий, арозходяться **номери**, під якими вони записані, — і цього вже досить, щоб модель,навчена з першим словником, не працювала з другим. На менших обсягах розходитьсяй сам набір.Ось чому в лекції числа WordPiece не наводяться: те, чого не можна відтворити,цитувати не можна. Там WordPiece показано на словнику, зробленому **руками** — вінвідтворюється точно, і поділ на ньому читач може перевірити на папері.І ось чому токенізатор **передають файлом словника** разом із моделлю, а неописом алгоритму: рецепт відтворює процедуру, але не результат.## 4 · Словник моделіДалі ми навчатимемо модель **на цілих словах**, а не на шматках WordPiece — і цесвідоме рішення. Причина в попередній клітинці: словник шматків між прогонамитрохи різний, а нам потрібно, щоб числа зошита відтворювались. Механізм BERT відцього не міняється: маскується токен, який би він не був.Слова, що трапились рідше пʼяти разів, зводимо в `[UNK]`.

In [ ]:
PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4

counts = Counter(w for r in train_rows for w in r[2])
words = SPECIAL + sorted(w for w, n in counts.items() if n >= 5)
vocab = {w: i for i, w in enumerate(words)}
V = len(vocab)

def encode(rs):
    return [[vocab.get(w, UNK) for w in r[2]] for r in rs]

train_ids = encode(train_rows)
test_ids = encode(test_rows)
train_program = np.array([r[0] for r in train_rows])

unk_share = np.mean([t == UNK for s in test_ids for t in s])
print('слів у словнику моделі', V)
print('спецтокенів у ньому   ', len(SPECIAL))
print('частка [UNK] на перевірній частині: %.4f' % unk_share)

## 5 · Вхід BERT: пара речень, сегменти й спецтокениОдин навчальний приклад BERT — це **дві** послідовності, складені в один рядок:```[CLS] перше речення [SEP] друге речення [SEP]```плюс масив **сегментів** тієї самої довжини: нулі на першому реченні разом ізйого `[SEP]`, одиниці на другому.У нашому корпусі немає документів із порядком речень, тож «наступним» миназиваємо **наступний рядок того самого каталогу програми**, а «випадковим» —рядок звідусіль. Це чесний аналог: позитивна пара з одного домену, негативна —з різних.Складімо один приклад і надрукуймо **розміри всіх тензорів** — це те саме, щорозібрано в лекції.

In [ ]:
MAX_LEN = 32

def make_pairs(sent_ids, program_of, rng, n_pairs):
    '''Індекси пар речень і мітка «друге є наступним».'''
    n = len(sent_ids)
    a_idx = rng.integers(0, n - 1, size=n_pairs)
    is_next = rng.integers(0, 2, size=n_pairs).astype(bool)
    b_idx = np.where(is_next, a_idx + 1, rng.integers(0, n, size=n_pairs))
    # «наступним» вважаємо лише сусіда з тієї самої програми
    same = program_of[a_idx] == program_of[np.minimum(a_idx + 1, n - 1)]
    is_next = is_next & same
    b_idx = np.where(is_next, a_idx + 1, b_idx)
    return a_idx, b_idx, is_next

def build_batch(sent_ids, a_idx, b_idx, is_next, max_len=MAX_LEN):
    '''Складає [CLS] A [SEP] B [SEP], сегменти й заповнювач.'''
    rows_ids, rows_seg = [], []
    for a, b in zip(a_idx, b_idx):
        first, second = sent_ids[a], sent_ids[b]
        room = max_len - 3                       # три спецтокени завжди на місці
        while len(first) + len(second) > room:   # ріжемо довше з двох
            if len(first) >= len(second):
                first = first[:-1]
            else:
                second = second[:-1]
        ids = [CLS] + list(first) + [SEP] + list(second) + [SEP]
        seg = [0] * (len(first) + 2) + [1] * (len(second) + 1)
        rows_ids.append(ids + [PAD] * (max_len - len(ids)))
        rows_seg.append(seg + [0] * (max_len - len(seg)))
    return (torch.tensor(rows_ids, dtype=torch.long),
            torch.tensor(rows_seg, dtype=torch.long),
            torch.tensor(np.asarray(is_next), dtype=torch.long))

show = np.random.default_rng(3)
a, b, nx = make_pairs(train_ids, train_program, show, 1)
ids_one, seg_one, nsp_one = build_batch(train_ids, a, b, nx)

back = {i: w for w, i in vocab.items()}
real = int((ids_one[0] != PAD).sum())
print('токени :', ' '.join(back[int(t)] for t in ids_one[0][:real]))
print('сегменти:', ' '.join(str(int(s)) for s in seg_one[0][:real]))
print('мітка «друге є наступним»:', int(nsp_one[0]))
print()
print('розмір масиву номерів   ', tuple(ids_one.shape))
print('розмір масиву сегментів ', tuple(seg_one.shape))

## 6 · Псування входу: схема 80-10-10Обираємо 15 % позицій (спецтокени не чіпаємо) і псуємо їх у трьох різнихспособах:- **80 %** замінюємо на `[MASK]`;- **10 %** замінюємо на випадкове слово;- **10 %** лишаємо як є.Позиції, які треба передбачити, запамʼятовуємо в окремому масиві `targets`;скрізь інде там стоїть `-100` — число, яке функція втрати torch пропускає.

In [ ]:
def corrupt(ids, rng, vocab_size, p_mask=0.15,
            share_mask=0.8, share_random=0.1):
    '''Псує ~15 % позицій. Повертає зіпсований вхід і те, що треба вгадати.'''
    ids = ids.clone()
    scored = (ids != PAD) & (ids != CLS) & (ids != SEP)
    draw = torch.tensor(rng.random(ids.shape), dtype=torch.float)
    chosen = scored & (draw < p_mask)

    targets = torch.full_like(ids, -100)          # -100 = «цю позицію не рахуємо»
    targets[chosen] = ids[chosen]

    what = torch.tensor(rng.random(ids.shape), dtype=torch.float)
    to_mask = chosen & (what < share_mask)
    to_rand = chosen & (what >= share_mask) & (what < share_mask + share_random)
    ids[to_mask] = MASK
    if to_rand.any():
        # випадкове слово беремо зі звичайних, а не зі спецтокенів
        ids[to_rand] = torch.tensor(
            rng.integers(len(SPECIAL), vocab_size, size=int(to_rand.sum())),
            dtype=torch.long)
    return ids, targets

demo_rng = np.random.default_rng(11)
noisy_one, targets_one = corrupt(ids_one, demo_rng, V, p_mask=0.5)   # 50 %, щоб було видно
print('було :', ' '.join(back[int(t)] for t in ids_one[0][:real]))
print('стало:', ' '.join(back[int(t)] for t in noisy_one[0][:real]))
print('треба вгадати на позиціях:',
      [i for i in range(real) if int(targets_one[0][i]) != -100])

## 7 · МодельНаш BERT — це те саме, що описано в лекції, тільки маленьке: ширина 128, двашари, дві голови, мережа прямого проходу на 256.Три таблиці ембедингів складаються поелементно, далі стос енкодерів, далі двіголови: MLM (відновити слово) і NSP (чи друге речення справді наступне).

In [ ]:
class TinyBert(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=2, n_heads=2,
                 d_ff=256, max_len=MAX_LEN, dropout=0.1):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.seg = nn.Embedding(2, d_model)          # два сегменти: A і B
        self.pos = nn.Embedding(max_len, d_model)    # позиції вчаться, як у BERT
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout=dropout,
                                           activation='gelu', batch_first=True,
                                           norm_first=False)   # післянормалізація
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.mlm_dense = nn.Linear(d_model, d_model)
        self.mlm_norm = nn.LayerNorm(d_model)
        self.mlm_out = nn.Linear(d_model, vocab_size)
        self.mlm_out.weight = self.tok.weight        # звʼязані ваги: одна таблиця
        self.pooler = nn.Linear(d_model, d_model)
        self.nsp_out = nn.Linear(d_model, 2)
        self.apply(self._init)

    @staticmethod
    def _init(module):
        # як у BERT: малий розкид. Без цього звʼязані ваги дають велетенські
        # оцінки на старті, і перші сотні кроків ідуть на їх приборкання
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)
        if isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight); nn.init.zeros_(module.bias)

    def encode(self, ids, seg):
        positions = torch.arange(ids.shape[1], device=ids.device).unsqueeze(0)
        x = self.tok(ids) + self.seg(seg) + self.pos(positions)   # три таблиці
        x = self.drop(self.norm(x))
        return self.encoder(x, src_key_padding_mask=ids.eq(PAD))

    def mlm_scores(self, h):
        return self.mlm_out(self.mlm_norm(torch.nn.functional.gelu(self.mlm_dense(h))))

    def nsp_scores(self, h):
        return self.nsp_out(torch.tanh(self.pooler(h[:, 0])))     # [CLS] — нульова позиція

probe_model = TinyBert(V)
print('ваг у моделі разом із головами:', sum(p.numel() for p in probe_model.parameters()))

Тепер найцікавіше в цьому розділі — **розміри всіх тензорів на одномуприкладі**. Проженемо наш єдиний приклад крізь модель і надрукуємо форму накожному кроці.Одне зауваження про друге число у формах. Рядок добивається заповнювачем до`MAX_LEN`, тож позицій завжди 32, скільки б слів не було в реченні. Це вимогаматричних обчислень: пачка має бути прямокутною. Змістовних позицій у нашомуприкладі менше — їх надруковано вище.

In [ ]:
fresh_rng = np.random.default_rng(21)
noisy_fresh, targets_fresh = corrupt(ids_one, fresh_rng, V, p_mask=0.15)

with torch.no_grad():
    probe_model.eval()
    n_pos = ids_one.shape[1]
    positions = torch.arange(n_pos).unsqueeze(0)
    e_tok = probe_model.tok(ids_one)
    e_seg = probe_model.seg(seg_one)
    e_pos = probe_model.pos(positions)
    summed = e_tok + e_seg + e_pos
    hidden = probe_model.encode(noisy_fresh, seg_one)
    picked = targets_fresh != -100
    scores = probe_model.mlm_scores(hidden[picked])
    nsp = probe_model.nsp_scores(hidden)

for name, tensor in [('номери токенів', ids_one), ('номери сегментів', seg_one),
                     ('ембединги слів', e_tok), ('ембединги сегментів', e_seg),
                     ('ембединги позицій', e_pos), ('сума трьох', summed),
                     ('вихід стосу енкодерів', hidden),
                     ('вибрані позиції', hidden[picked]),
                     ('оцінки по словнику', scores),
                     ('оцінка NSP', nsp)]:
    print('%-24s %-16s %9d чисел' % (name, tuple(tensor.shape), tensor.numel()))

del probe_model
gc.collect()

Зверни увагу на передостанній рядок. Оцінки по словнику рахуються **лише** навибраних позиціях, а не на всіх тридцяти двох: це найширший тензор моделі, ірахувати його там, де відповіді все одно немає, — марна робота.## 8 · ПередтренуванняШвидкість навчання ми дібрали окремим прогоном: сітка `0.0005 … 0.008`,перплексія на **відкладеній** частині, мінімум усередині сітки. Тут беремодібране значення й не підглядаємо в перевірну.

In [ ]:
STEPS = 500          # свідоме обмеження заради бюджету зошита
BATCH = 64
LR = 0.001           # дібрано на відкладеній частині окремим прогоном

def pretrain(seed, shares=(0.8, 0.1), use_nsp=True, steps=STEPS):
    '''Одне передтренування. shares — частки [MASK] і випадкового слова.'''
    torch.manual_seed(seed)
    rng = np.random.default_rng(1000 + seed)
    model = TinyBert(V)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps,
                                                pct_start=0.1)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    model.train()
    for step in range(steps):
        a, b, nx = make_pairs(train_ids, train_program, rng, BATCH)
        ids, seg, y_nsp = build_batch(train_ids, a, b, nx)
        noisy, targets = corrupt(ids, rng, V, 0.15, shares[0], shares[1])
        hidden = model.encode(noisy, seg)
        picked = targets != -100
        loss = loss_fn(model.mlm_scores(hidden[picked]), targets[picked])
        if use_nsp:
            loss = loss + loss_fn(model.nsp_scores(hidden), y_nsp)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
    model.eval()
    return model

t0 = time.process_time()
bert_80_10_10 = pretrain(seed=0, shares=(0.8, 0.1))
print('передтренування 80-10-10: %.1f с процесорних на %d кроків' %
      (time.process_time() - t0, STEPS))

## 9 · Що модель навчилась відновлюватиНайпряміша перевірка: візьмімо справжні речення, заховаємо в них по слову йподивимось, що модель пропонує.

In [ ]:
@torch.no_grad()
def fill_gap(model, sentence_words, gap_index, top=5):
    '''Ховає одне слово й повертає найімовірніші заміни.'''
    ids = [CLS] + [vocab.get(w, UNK) for w in sentence_words] + [SEP]
    ids[gap_index + 1] = MASK               # +1, бо нульова позиція — [CLS]
    ids = ids + [PAD] * (MAX_LEN - len(ids))
    ids_t = torch.tensor([ids])
    seg_t = torch.zeros_like(ids_t)
    hidden = model.encode(ids_t, seg_t)
    scores = model.mlm_scores(hidden[0, gap_index + 1])
    best = torch.topk(scores, top).indices.tolist()
    return [back[i] for i in best]

examples = [(['не', 'вдалося', 'зберегти', 'файл'], 2),
            (['помилка', 'доступу', 'до', 'файлу'], 0),
            (['натисніть', 'кнопку', 'щоб', 'продовжити'], 1)]

for sentence_words, gap in examples:
    shown = list(sentence_words)
    shown[gap] = '___'
    print(' '.join(shown))
    print('   модель пропонує:', ', '.join(fill_gap(bert_80_10_10, sentence_words, gap)))
    print('   насправді там  :', sentence_words[gap])
    print()

Пропозиції не блискучі, і прикидатись не будемо: на пʼятистах пачках модельздебільшого вивчила **частоти** — у списку стоять найчастіші слова корпусу.Але подекуди вже працює й контекст: у другому прикладі правильне словопотрапило в пʼятірку. Саме так виглядає передтренування на початку: спершучастоти, потім контекст.## 10 · Дослід: що буде, якщо завжди ставити `[MASK]`Тепер перевіримо те, заради чого в BERT існує схема 80-10-10. Навчимо **другу**модель, яка на кожній обраній позиції ставить `[MASK]` і ніколи не лишає словона місці, — і поміряємо обидві двома різними способами:- **точність на `[MASK]`**: позицію заховано, треба назвати слово;- **точність на неторканому слові**: слово стоїть на місці, треба назвати його ж.Друге завдання виглядає безглуздо простим. У цьому й суть: модель, яка ніколи небачила «неторканих» позицій під передбаченням, його провалює — а саме такіпозиції їй трапляться при донавчанні, де `[MASK]` немає взагалі.

In [ ]:
@torch.no_grad()
def accuracy(model, sent_ids, program_of, mode, seed=999, batches=8):
    '''mode='mask' — позицію заховано; mode='keep' — слово лишили на місці.'''
    rng = np.random.default_rng(seed)
    right = total = 0
    for _ in range(batches):
        a, b, nx = make_pairs(sent_ids, program_of, rng, BATCH)
        ids, seg, _ = build_batch(sent_ids, a, b, nx)
        shares = (1.0, 0.0) if mode == 'mask' else (0.0, 0.0)
        noisy, targets = corrupt(ids, rng, V, 0.15, shares[0], shares[1])
        hidden = model.encode(noisy, seg)
        picked = targets != -100
        right += int((model.mlm_scores(hidden[picked]).argmax(-1) == targets[picked]).sum())
        total += int(picked.sum())
    return right / total

test_program = np.array([r[0] for r in test_rows])

t0 = time.process_time()
bert_100_0_0 = pretrain(seed=0, shares=(1.0, 0.0))
print('передтренування «завжди [MASK]»: %.1f с' % (time.process_time() - t0))
print()

print('%-22s%16s%22s' % ('схема псування', 'точність [MASK]', 'точність на слові'))
for name, model in [('80-10-10', bert_80_10_10), ('завжди [MASK]', bert_100_0_0)]:
    on_mask = accuracy(model, test_ids, test_program, 'mask')
    on_keep = accuracy(model, test_ids, test_program, 'keep')
    print('%-22s%16.4f%22.4f' % (name, on_mask, on_keep))

Порівняй два стовпчики. Ліворуч — те, чого обидві моделі вчились однаковостаранно, і різниці там майже немає. Праворуч — те, чого одна з них не вчиласьніколи.Це й є відповідь на питання «навіщо десять відсотків незмінених слів»: вони нероблять маскування кращим, вони роблять осмисленим **подання неторканогослова**. А саме такі позиції трапляються при донавчанні, де `[MASK]` не буває.⚠️ Тут одне зерно, тож окремі цифри гуляють. У лекції те саме порівняннязроблено на **пʼятьох** зернах і для чотирьох схем псування; там видно, якірізниці більші за розкид, а які ні.## 11 · Заморожений енкодер плюс логістична регресіяОстанній крок — власне задача. Беремо передтреновану модель, **нічого в ній неміняємо**, беремо середнє по векторах слів як подання рядка й учимо на цихвекторах звичайну логістичну регресію.Порівняємо чотири варіанти: дві наші моделі, той самий енкодер **без жодногонавчання** (щоб побачити, що дало саме передтренування) і TF-IDF із теми 06.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score

@torch.no_grad()
def sentence_vectors(model, sent_ids, batch=256):
    '''Подання рядка: середнє по його словах, без [CLS] і [SEP].'''
    out = []
    for start in range(0, len(sent_ids), batch):
        chunk = sent_ids[start:start + batch]
        rows_ids = []
        for s in chunk:
            ids = [CLS] + list(s)[:MAX_LEN - 2] + [SEP]
            rows_ids.append(ids + [PAD] * (MAX_LEN - len(ids)))
        ids_t = torch.tensor(rows_ids, dtype=torch.long)
        hidden = model.encode(ids_t, torch.zeros_like(ids_t))
        keep = ((ids_t != PAD) & (ids_t != CLS) & (ids_t != SEP)).unsqueeze(-1).float()
        out.append(((hidden * keep).sum(1) / keep.sum(1).clamp(min=1)).numpy())
    return np.concatenate(out)

N_LABELS = 8000                       # стільки міток даємо зонду
label_rng = np.random.default_rng(500)
chosen = label_rng.choice(len(train_ids), N_LABELS, replace=False).tolist()
y_train = train_labels[chosen]
train_subset = [train_ids[i] for i in chosen]

torch.manual_seed(77)
bert_untrained = TinyBert(V).eval()    # той самий енкодер, який нічого не вчив

results = {}
for name, model in [('BERT 80-10-10', bert_80_10_10),
                    ('BERT завжди [MASK]', bert_100_0_0),
                    ('енкодер без навчання', bert_untrained)]:
    X_train = sentence_vectors(model, train_subset)
    X_test = sentence_vectors(model, test_ids)
    clf = LogisticRegression(max_iter=2000).fit(X_train, y_train)
    results[name] = f1_score(test_labels, clf.predict(X_test))

# рубіж із теми 06: TF-IDF по тих самих рядках і тих самих мітках
vectorizer = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
X_train_tfidf = vectorizer.fit_transform(' '.join(train_rows[i][2]) for i in chosen)
X_test_tfidf = vectorizer.transform(' '.join(r[2]) for r in test_rows)
tfidf_clf = LogisticRegression(max_iter=2000).fit(X_train_tfidf, y_train)
results['TF-IDF + логістична'] = f1_score(test_labels, tfidf_clf.predict(X_test_tfidf))

print('%-24s%14s' % ('що взяли за ознаки', 'F1 «помилка»'))
for name, score in results.items():
    print('%-24s%14.4f' % (name, score))

Що тут варто прочитати з чисел — і чого читати не варто.**Порівняй передтреновані моделі з енкодером без навчання.** Різниця між нимиі є те, що приніс масковий обʼєктив: та сама будова, ті самі ваги за формою,але одна з них прочитала корпус, а друга ні.**Не роби висновку про схеми псування з цієї таблиці.** Тут одне зерно, арізниця між схемами на зонді мала. У лекції це порівняння зроблено на пʼятьохзернах — і там видно, що на цій метриці схеми не розрізняються, тоді як наточності з розділу 10 розрізняються різко.**TF-IDF нас обганяє** — і це нормально для макета, який бачив пʼятсот пачок ікорпус у шість тисяч разів менший за той, на якому вчать справжній BERT.Передтренування виграє тоді, коли міток мало, а тексту багато; ми дали зондувісім тисяч міток, і на такій кількості лічильники ще сильні.## 12 · Скільки важить справжній BERTОстаннє — арифметика, яку варто вміти робити самому. Формула кількості вагпроста, і ми звіримо її з тим, що дає `torch`, коли справді побудувати модельтаких розмірів.

In [ ]:
def bert_parameters(V, H, L, F, P):
    '''Скільки ваг у BERT заданих розмірів. Кожен доданок перевіряється на папері.'''
    embeddings = V * H + P * H + 2 * H + 2 * H     # слова, позиції, сегменти, LayerNorm
    attention = 4 * (H * H + H) + 2 * H            # Q, K, V, вихід і LayerNorm
    feedforward = (F * H + F) + (H * F + H) + 2 * H
    pooler = H * H + H
    return dict(words=V * H, positions=P * H, segments=2 * H,
                embeddings=embeddings, layer=attention + feedforward,
                total=embeddings + L * (attention + feedforward) + pooler)

def build_and_count(V, H, L, A, F, P):
    '''Те саме, але справжньою побудовою в torch.'''
    class Skeleton(nn.Module):
        def __init__(self):
            super().__init__()
            self.tok = nn.Embedding(V, H); self.pos = nn.Embedding(P, H)
            self.seg = nn.Embedding(2, H); self.norm = nn.LayerNorm(H)
            layer = nn.TransformerEncoderLayer(H, A, F, activation='gelu',
                                               batch_first=True)
            self.enc = nn.TransformerEncoder(layer, L, enable_nested_tensor=False)
            self.pooler = nn.Linear(H, H)
    model = Skeleton()
    n = sum(p.numel() for p in model.parameters())
    del model; gc.collect()
    return n

CONFIGS = [('наш макет', V, 128, 2, 2, 256, MAX_LEN),
           ('BERT-base', 30522, 768, 12, 12, 3072, 512),
           ('BERT-large', 30522, 1024, 24, 16, 4096, 512)]

print('%-12s%13s%11s%11s%13s%16s%10s' % ('модель', 'таб. слів', 'позиції',
                                          'сегменти', 'один шар', 'усього', 'МіБ'))
for name, v, h, l, a, f, p in CONFIGS:
    counted = bert_parameters(v, h, l, f, p)
    in_torch = build_and_count(v, h, l, a, f, p)
    assert in_torch == counted['total'], (name, in_torch, counted['total'])
    print('%-12s%13d%11d%11d%13d%16d%10.1f'
          % (name, counted['words'], counted['positions'], counted['segments'],
             counted['layer'], counted['total'], counted['total'] * 4 / 1048576))
print()
print('✅ формула збігається з torch до одиниці на всіх трьох конфігураціях')

small = bert_parameters(V, 128, 2, 256, MAX_LEN)['total']
big = bert_parameters(30522, 768, 12, 3072, 512)['total']
print()
print('BERT-base більший за наш макет у %.2f раза' % (big / small))
print('на навчання з Adam BERT-base просить %.2f ГіБ лише під ваги й моменти'
      % (big * 16 / 1073741824))

## 13 · Скільки це коштувалоОстанньою клітинкою — власний час зошита. Порівняй його з тим, що обіцяно напочатку: якщо розбіжність велика, значить, машина зайнята чимось іще.

In [ ]:
del bert_80_10_10, bert_100_0_0, bert_untrained
gc.collect()

spent = time.process_time() - ЗАПУСК
print('процесорного часу зошита: %.1f с (%.1f хв)' % (spent, spent / 60))
print('навантаження машини зараз:', round(os.getloadavg()[0], 2))
print('потоків у torch          :', torch.get_num_threads())

## Що спробувати самому**🟢 Рівень 1.** Зміни `STEPS` із 500 на 250 і на 1000 і подивись, як їдутьобидві точності з розділу 10. Питання, на яке треба відповісти: розрив міжсхемами росте чи спадає з навчанням?**🟡 Рівень 2.** Додай у розділ 11 четверту модель — передтреновану **без** NSP(параметр `use_nsp=False` у `pretrain`). Порівняй F1 зонда. Не роби висновку зодного зерна: прожени три й подивись, чи купи перетинаються.**🔴 Рівень 3.** Замини словник цілих слів на словник WordPiece із розділу 3 іперероби `encode`. Головне питання: чи зменшиться частка `[UNK]` на перевірнійчастині — і чи виграє від цього зонд.Детальніші завдання з критеріями — у [homework.html](homework.html).